# Create Dust_temp.in files
Store each file in its respective climate input directory on s3, which means it'll get copied over to the run directory. (means we need to have already created the climate input dir!)

## Read in dust flux timeseries by site 

In [1]:
import math
import os 
from typing import Optional, List

import fsspec
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import xarray as xr 

from ew_workflows import scepter_helperFxns as shf

In [8]:
# --- where to save outputs ----------
#
#  ----------------------------------------------------------------
# [ RUN NAMES ]
# exe_central1p5 ; exe_central2p0 ; exe_limBio1p5 ; exe_lowCCS1p5 ; 
# exe_lowCDR1p5 ; exe_noBiochar1p5 ; exe_noWeathering1p5 ; 
# exe_ref1p5 ; exe_techBehaviorChange1p5 ; exe_techBehaviorChange2p0 ;
# 
# (updated dust)
# exe_central1p5_max10tha   exe_noWeathering1p5_max5tha
# 
# [ RUNS TO COMPARE ]
# exe_central1p5 ; exe_noWeathering1p5 ; exe_techBehaviorChange1p5
# ----------------------------------------------------------------
#
#
# --- add crop area options (it will save multiple identical clim dirs,
#     but rock-application-specific files will be added later)
REPEAT_CLIM_FOR_CROP_AREA = False
crop_area_frac_arr = np.round(np.linspace(0.1, 1, 10), 2) # from 10% to 100% in 10% increments
# --- add application rate options 
REPEAT_CLIM_FOR_APP_RATE = True 
USE_FIXED_APP_RATE = True
app_rate_arr = np.array([0.25, 0.75]) # np.array([0.05, 0.1, 0.5, 1., 2.5])  # np.array([5, 7.5, 10, 15, 20, 30, 40, 50])  
# -------------------------------------------------------------------
#
#
# <Site>_<deg_resolution>_<climtype>
RUNNAME = "exe_techBehaviorChange1p5"
climtype = "yearly_ltm"
# ------------------------------------
LOCATION_DATASETS = {
    "largest_cropland": {
        "label": "largestCropland",
    },
    "wettest_crop25pct": {
        "label": "wettestSite_crop25pct",
    },
    "repcells1": {
        "label": "repcells1",
    }
}

# --- select which location dataset to use
LOCATION_KEY = "repcells1" # "largest_cropland" ; "wettest_crop25pct" ; "repcells1"
loc = LOCATION_DATASETS[LOCATION_KEY]
site_label = loc["label"] # for saving

In [9]:
# # --- make any modifications
# # -----------------------------------------------------------------
# # [ MAX 10 T/HA ]
# # df["rockflx_t_ha"] = df["rockflx_t_ha"].clip(upper=10)
# # df["rockflx_Mt_ha"] = df["rockflx_Mt_ha"].clip(upper=10e-6)
# # df["rockflx_Mt"] = np.nan

# # [ MAX 5 T/HA ]
# df["rockflx_t_ha"] = df["rockflx_t_ha"].clip(upper=5)
# df["rockflx_Mt_ha"] = df["rockflx_Mt_ha"].clip(upper=5e-6)
# df["rockflx_Mt"] = np.nan

# # [ ROCK AMOUNT MULTIPLIER ]
# rock_multiplier = 2
# df["rockflx_t_ha"] = df["rockflx_t_ha"] * rock_multiplier
# df["rockflx_Mt_ha"] = df["rockflx_Mt_ha"] * rock_multiplier
# df["rockflx_Mt"] = np.nan
# # -----------------------------------------------------------------
# df

In [10]:
# ---------------------------------------
# [ set up function inputs ]
t_add = 0.1
taudust = 0.05
output_filename = "Dust_temp.in"
dryrun = False
# ---------------------------------------
if REPEAT_CLIM_FOR_CROP_AREA:
        for frac in crop_area_frac_arr:
                #
                RUN_SUFFIX = f"{frac}-cropFraction"
                #
                # --- read in data
                deg_resolution = 1
                datpath = f's3://carbonplan-carbon-removal/ew-workflows-data/gcam/apprate_calculated'
                subdir = f'one-cell-per-region/{RUNNAME}/{RUN_SUFFIX}'
                fn_df = f'annual_rockflx_densest-ag-region_{deg_resolution}deg.pkl'
                fn_ds = f'cropland+gcamRegions_time_{deg_resolution}deg.zarr'

                # dataframe 
                df = pd.read_pickle(os.path.join(datpath, subdir, fn_df))
                all_regions = df['region_nospaces'].unique()

                # --- create output
                climpath_main = f"s3://carbonplan-carbon-removal/ew-workflows-data/scepter/clim/one-cell-per-region_1deg/{RUNNAME}/{RUN_SUFFIX}"
                for this_region in all_regions:
                        print(this_region)
                        # pull out this region 
                        dfreg = df[df['region_nospaces'] == this_region].copy()
                        # create output dir
                        outdir = os.path.join(climpath_main, f'{this_region}_{deg_resolution}_{climtype}')
                        climpath = outdir # read and write to same dir

                        # create Dust_temp.in file
                        outdict = shf.create_dust_input_from_gcam_df(
                                outdir,
                                climpath,
                                df = dfreg,
                                t_add = t_add,
                                taudust = taudust,
                                output_filename = "Dust_temp.in",
                                dryrun = dryrun,
                        )

elif REPEAT_CLIM_FOR_APP_RATE:
        for apprate in app_rate_arr:
                #
                if USE_FIXED_APP_RATE:
                        RUN_SUFFIX = f"{apprate}-tHaYr-appRate-FIXED"
                else:
                        RUN_SUFFIX = f"{apprate}-tHaYr-appRate"
                #
                # --- read in data
                deg_resolution = 1
                datpath = f's3://carbonplan-carbon-removal/ew-workflows-data/gcam/apprate_calculated'
                subdir = f'one-cell-per-region/{RUNNAME}/{RUN_SUFFIX}'
                fn_df = f'annual_rockflx_densest-ag-region_{deg_resolution}deg.pkl'
                fn_ds = f'cropland+gcamRegions_time_{deg_resolution}deg.zarr'

                # dataframe 
                df = pd.read_pickle(os.path.join(datpath, subdir, fn_df))
                all_regions = df['region_nospaces'].unique()

                # --- create output
                climpath_main = f"s3://carbonplan-carbon-removal/ew-workflows-data/scepter/clim/one-cell-per-region_1deg/{site_label}/{RUNNAME}/{RUN_SUFFIX}"
                for this_region in all_regions:
                        print(this_region)
                        # pull out this region 
                        dfreg = df[df['region_nospaces'] == this_region].copy()
                        # create output dir
                        outdir = os.path.join(climpath_main, f'{this_region}_{deg_resolution}_{climtype}')
                        climpath = outdir # read and write to same dir

                        # create Dust_temp.in file
                        outdict = shf.create_dust_input_from_gcam_df(
                                outdir,
                                climpath,
                                df = dfreg,
                                t_add = t_add,
                                taudust = taudust,
                                output_filename = "Dust_temp.in",
                                dryrun = dryrun,
                        )

else:
        # --- read in data
        deg_resolution = 1
        datpath = f's3://carbonplan-carbon-removal/ew-workflows-data/gcam/apprate_calculated'
        subdir = f'one-cell-per-region/{RUNNAME}/{RUN_SUFFIX}'
        fn_df = f'annual_rockflx_densest-ag-region_{deg_resolution}deg.pkl'
        fn_ds = f'cropland+gcamRegions_time_{deg_resolution}deg.zarr'

        # dataframe 
        df = pd.read_pickle(os.path.join(datpath, subdir, fn_df))
        all_regions = df['region_nospaces'].unique()
        #
        for this_region in all_regions:
                print(this_region)
                # pull out this region 
                dfreg = df[df['region_nospaces'] == this_region].copy()
                # create output dir
                outdir = os.path.join(climpath_main, f'{this_region}_{deg_resolution}_{climtype}')
                climpath = outdir # read and write to same dir

                # create Dust_temp.in file
                outdict = shf.create_dust_input_from_gcam_df(
                        outdir,
                        climpath,
                        df = dfreg,
                        t_add = t_add,
                        taudust = taudust,
                        output_filename = "Dust_temp.in",
                        dryrun = dryrun,
                )

USA
Africa_Eastern
Africa_Northern
Africa_Southern
Africa_Western
Australia_NZ
Brazil
Canada
Central_America_and_Caribbean
Central_Asia
China
EU-12
EU-15
Europe_Eastern
Europe_Non_EU
European_Free_Trade_Association
India
Indonesia
Japan
Mexico
Middle_East
Pakistan
Russia
South_Africa
South_America_Northern
South_America_Southern
South_Asia
South_Korea
Southeast_Asia
Taiwan
Argentina
Colombia
USA
Africa_Eastern
Africa_Northern
Africa_Southern
Africa_Western
Australia_NZ
Brazil
Canada
Central_America_and_Caribbean
Central_Asia
China
EU-12
EU-15
Europe_Eastern
Europe_Non_EU
European_Free_Trade_Association
India
Indonesia
Japan
Mexico
Middle_East
Pakistan
Russia
South_Africa
South_America_Northern
South_America_Southern
South_Asia
South_Korea
Southeast_Asia
Taiwan
Argentina
Colombia


In [11]:
# plt.plot(outdict['time_yr'], outdict['cumrate_gm2yr'])
# plt.plot(dfreg['time'], dfreg['rockflx_t_ha'].cumsum()*100)
# plt.plot(outdict['time_yr'], outdict['rate_gm2yr'])
#plt.xlim(70,80)
#plt.ylim(7e4,8.7e4)

In [12]:
# ---
# dfreg

# import pandas as pd

# dftest = pd.read_pickle('s3://carbonplan-carbon-removal/ew-workflows-data/gcam/apprate_calculated/one-cell-per-region/exe_central1p5/10.0-tHaYr-appRate-FIXED/annual_rockflx_densest-ag-region_1deg.pkl')
# dftest = pd.read_pickle('s3://carbonplan-carbon-removal/ew-workflows-data/gcam/apprate_calculated/one-cell-per-region/exe_central1p5/0.8-cropFraction/annual_rockflx_densest-ag-region_1deg.pkl')

In [ ]:
# dftest['region'].unique()

<ArrowStringArray>
[                            'USA',                  'Africa_Eastern',
                 'Africa_Northern',                 'Africa_Southern',
                  'Africa_Western',                    'Australia_NZ',
                          'Brazil',                          'Canada',
   'Central America and Caribbean',                    'Central Asia',
                           'China',                           'EU-12',
                           'EU-15',                  'Europe_Eastern',
                   'Europe_Non_EU', 'European Free Trade Association',
                           'India',                       'Indonesia',
                           'Japan',                          'Mexico',
                     'Middle East',                        'Pakistan',
                          'Russia',                    'South Africa',
          'South America_Northern',          'South America_Southern',
                      'South Asia',                     'S

In [ ]:
# dftest[dftest['region'] == "South America_Southern"]

,reg_id,region,time,rockflx_t_ha,rockflx_Mt_ha,rockflx_Mt,lat,lon,gcam_gross_cdr_mtCO2,region_nospaces
2025,26,South America_Southern,2020,0.000000,0.000000e+00,0.000000,-24.5,-54.5,0.000000,South_America_Southern
2026,26,South America_Southern,2021,0.169122,1.691217e-07,0.082417,-24.5,-54.5,0.825154,South_America_Southern
2027,26,South America_Southern,2022,0.338243,3.382434e-07,0.164833,-24.5,-54.5,1.650308,South_America_Southern
2028,26,South America_Southern,2023,0.507365,5.073651e-07,0.247250,-24.5,-54.5,2.475462,South_America_Southern
2029,26,South America_Southern,2024,0.676487,6.764868e-07,0.329667,-24.5,-54.5,3.300616,South_America_Southern
...,...,...,...,...,...,...,...,...,...,...
2101,26,South America_Southern,2096,20.908648,2.090865e-05,10.189240,-24.5,-54.5,102.014428,South_America_Southern
2102,26,South America_Southern,2097,20.909008,2.090901e-05,10.189416,-24.5,-54.5,102.016182,South_America_Southern
2103,26,South America_Southern,2098,20.909367,2.090937e-05,10.189591,-24.5,-54.5,102.017937,South_America_Southern
2104,26,South America_Southern,2099,20.909727,2.090973e-05,10.189766,-24.5,-54.5,102.019691,South_America_Southern
